# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/vedant08mehta/Flyrank-assignment1.git
%cd /content/Flyrank-assignment1

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 136 (delta 45), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.88 MiB | 9.77 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/Flyrank-assignment1


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row represents one content page/content item. The performance variables are aggregated over a 90-day window, while fields such as content_age_days and days_since_last_update provide additional page-level context. The target is whether the page is showing a declining trend during the measured period.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")

print(
    f"Duplicate content_id rows: "
    f"{df['content_id'].duplicated().sum():,}"
)

print("\n90-day performance fields:")
print([
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d"
])

Rows: 30,000
Columns: 44
Unique content_id values: 30,000
Duplicate content_id rows: 0

90-day performance fields:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, clicks_90d, ctr, avg_position, engagement_rate, content_age_days, days_since_last_update, word_count, search_volume, competition, and content_type. These are candidate predictors available for analysis before the target is evaluated.

Label: is_declining_label, representing whether trend_direction is "down".

Context: content_id, client_id, main_intent, content_type, age_tier, freshness_tier, and position_tier. These describe the content and its context and may be useful for grouping, interpretation, or later analysis.

Excluded: trend_direction and trend_pct are excluded from the model because they describe the outcome used to construct the target and would leak information about the label. is_declining_label is also excluded from the feature set because it is the target itself.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "search_volume",
    "competition",
    "content_type"
]

label_col = "is_declining_label"

context_cols = [
    "content_id",
    "client_id",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "position_tier"
]

excluded_cols = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Features:")
print(feature_cols)

print("\nLabel:")
print(label_col)

print("\nContext:")
print(context_cols)

print("\nExcluded:")
print(excluded_cols)

Features:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'search_volume', 'competition', 'content_type']

Label:
is_declining_label

Context:
['content_id', 'client_id', 'main_intent', 'age_tier', 'freshness_tier', 'position_tier']

Excluded:
['trend_direction', 'trend_pct', 'is_declining_label']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
print("=== Grain check ===")

print(f"Total rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(
    f"Duplicate content_id rows: "
    f"{df['content_id'].duplicated().sum():,}"
)


# Create the label from the observed outcome
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


print("\n=== Missing-value check ===")

check_cols = feature_cols + [label_col]

missing = df[check_cols].isna().sum()

print(
    missing[missing > 0].sort_values(ascending=False)
)


print("\n=== Target distribution ===")

print(
    df[label_col].value_counts(dropna=False)
)

print(
    f"Declining rate: "
    f"{df[label_col].mean():.3f}"
)


print("\n=== 90-day field check ===")

for col in [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d"
]:
    print(
        f"{col}: "
        f"min={df[col].min():,.0f}, "
        f"max={df[col].max():,.0f}, "
        f"missing={df[col].isna().sum():,}"
    )

=== Grain check ===
Total rows: 30,000
Unique content_id: 30,000
Duplicate content_id rows: 0

=== Missing-value check ===
word_count       7699
search_volume    2468
competition      2468
dtype: int64

=== Target distribution ===
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Declining rate: 0.542

=== 90-day field check ===
impressions_90d: min=1, max=517,715, missing=0
clicks_90d: min=0, max=4,178, missing=0
pageviews_90d: min=0, max=5,998, missing=0
sessions_90d: min=1, max=4,345, missing=0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot establish causation or explain Google's ranking algorithm. The 90-day aggregates also hide day-to-day variation, so they cannot show exactly when a change happened. Coverage may differ between pages and clients, and some pages may have limited search history. The 90-day fields are also overlapping summaries rather than independent observations across time. Therefore, results should be treated as observed and directional decision-support evidence rather than causal proof. Client-level differences may also affect generalization, so later validation should keep clients separated between training and testing.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Data-limit checks:")

print(
    f"Pages with fewer than 10 impression days: "
    f"{(df['days_with_impressions'] < 10).sum():,}"
)

print(
    f"Pages with missing 90-day impressions: "
    f"{df['impressions_90d'].isna().sum():,}"
)

print(
    f"Pages with missing average position: "
    f"{df['avg_position'].isna().sum():,}"
)

print(
    f"Pages with missing content age: "
    f"{df['content_age_days'].isna().sum():,}"
)

Data-limit checks:
Pages with fewer than 10 impression days: 4,408
Pages with missing 90-day impressions: 0
Pages with missing average position: 0
Pages with missing content age: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.